In [1]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import scipy.io.wavfile as wav
from scipy.optimize import differential_evolution
import copy
from IPython.display import display, Audio
import os
import sys

# Import your physical model components
sys.path.append(os.path.abspath(os.path.join('..')))
from config import SR, MEMBRANE, HAMMER, SYSTEM
from synthesizer import TonbakSynthesizer

In [2]:
target_wav_path = "../../single_stroke_analysis/data/raw/hit_low/GIO_MEE_tombak_hit_low.wav"
target_y, _ = librosa.load(target_wav_path, sr=SR)

# Trim/pad to exactly 1.0 second for consistent analysis
target_length = int(SR * 1.0)
if len(target_y) > target_length:
    target_y = target_y[:target_length]
else:
    target_y = np.pad(target_y, (0, target_length - len(target_y)))

print("Target Real Tonbak Recording:")
display(Audio(target_y, rate=SR))

Target Real Tonbak Recording:


In [3]:
# (Min, Max) search boundaries for the optimizer
BOUNDS = [
    (0.01, 0.12),      # [0] hit_r (Center to Rim)
    (0.005, 0.05),     # [1] hammer mass
    (1e4, 1.5e5),      # [2] hammer stiffness
    (1.0, 6.0),        # [3] hammer velocity
    (1e-4, 5e-4),      # [4] d3 (High-freq damping)
    (0.1, 0.8),        # [5] d1 (Low-freq damping)
    (1800.0, 2400.0)   # [6] tension_0 (Tuning)
]

def extract_features(audio_array, sr=SR):
    """Extracts the Log-Mel Spectrogram to capture timbre and decay contour."""
    mel_spec = librosa.feature.melspectrogram(y=audio_array, sr=sr, n_mels=64, hop_length=256)
    log_mel = librosa.power_to_db(mel_spec, ref=np.max)
    return log_mel

# Pre-calculate target features so we don't do it every loop
target_features = extract_features(target_y)

def objective_function(params):
    """Generates synthetic audio and returns the MSE against the target spectrogram."""
    hit_r, h_mass, h_stiff, h_vel, d3, d1, tension = params
    
    mem_cfg = copy.deepcopy(MEMBRANE)
    ham_cfg = copy.deepcopy(HAMMER)
    sys_cfg = copy.deepcopy(SYSTEM)
    
    # Inject optimizer parameters
    ham_cfg['mass'] = h_mass
    ham_cfg['stiffness'] = h_stiff
    ham_cfg['velocity_in'] = h_vel
    mem_cfg['d3'] = d3
    mem_cfg['d1'] = d1
    mem_cfg['tension_0'] = tension
    
    # Recalculate dependent variables
    mem_cfg['surface_density'] = mem_cfg['density'] * mem_cfg['thickness']
    mem_cfg['bending_stiffness'] = (mem_cfg['young_modulus'] * mem_cfg['thickness']**3) / (12 * (1 - mem_cfg['poisson_ratio']**2))
    
    synth = TonbakSynthesizer(
        hit_r=hit_r, hit_phi=0.0, pickup_r=0.08, pickup_phi=0.0,
        membrane_cfg=mem_cfg, hammer_cfg=ham_cfg, system_cfg=sys_cfg
    )
    
    try:
        synth_audio = synth.generate(duration_sec=1.0)
        synth_features = extract_features(synth_audio)
        loss = np.mean((target_features - synth_features) ** 2)
        return loss
    except Exception as e:
        return 1e6  # Heavy penalty if the physics engine explodes (NaNs)

In [4]:
print("Starting Differential Evolution Optimization...")

result = differential_evolution(
    objective_function, 
    bounds=BOUNDS, 
    strategy='best1bin', 
    maxiter=20,     # Generations (Increase to 40+ for better results)
    popsize=15,     # Candidates per generation
    mutation=(0.5, 1.0), 
    recombination=0.7,
    disp=True
)

best_params = result.x
print("\nOptimization Complete!")

Starting Differential Evolution Optimization...


KeyboardInterrupt: 

In [ ]:
# Unpack best parameters
hit_r, h_mass, h_stiff, h_vel, d3, d1, tension = best_params

print("--- Best Discovered Physics Configuration ---")
print(f"Hit Radius: {hit_r:.3f}")
print(f"Hammer Mass: {h_mass:.4f} kg")
print(f"Hammer Stiffness: {h_stiff:.1f} N/m")
print(f"Hammer Velocity: {h_vel:.2f} m/s")
print(f"Damping d3: {d3:.6f}")
print(f"Damping d1: {d1:.3f}")
print(f"Base Tension: {tension:.1f} N/m")

# Reconstruct the optimized synthetic sound
mem_cfg = copy.deepcopy(MEMBRANE)
ham_cfg = copy.deepcopy(HAMMER)
sys_cfg = copy.deepcopy(SYSTEM)

ham_cfg['mass'] = h_mass
ham_cfg['stiffness'] = h_stiff
ham_cfg['velocity_in'] = h_vel
mem_cfg['d3'] = d3
mem_cfg['d1'] = d1
mem_cfg['tension_0'] = tension
mem_cfg['surface_density'] = mem_cfg['density'] * mem_cfg['thickness']
mem_cfg['bending_stiffness'] = (mem_cfg['young_modulus'] * mem_cfg['thickness']**3) / (12 * (1 - mem_cfg['poisson_ratio']**2))

best_synth = TonbakSynthesizer(
    hit_r=hit_r, hit_phi=0.0, pickup_r=0.08, pickup_phi=0.0,
    membrane_cfg=mem_cfg, hammer_cfg=ham_cfg, system_cfg=sys_cfg
)
synth_y = best_synth.generate(duration_sec=1.0)
synth_features = extract_features(synth_y)

print("\nOptimized Synthetic Tonbak Strike:")
display(Audio(synth_y, rate=SR))

# Plot the visual comparison
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
librosa.display.specshow(target_features, sr=SR, hop_length=256, x_axis='time', y_axis='mel')
plt.title('Target Real Recording')
plt.colorbar(format='%+2.0f dB')

plt.subplot(1, 2, 2)
librosa.display.specshow(synth_features, sr=SR, hop_length=256, x_axis='time', y_axis='mel')
plt.title('Optimized Physical Model')
plt.colorbar(format='%+2.0f dB')

plt.tight_layout()
plt.show()